# Decision Trees from Scratch

In this notebook, we will build a fundamental component of XGBoost: a Decision Tree.
We will use pure Python and NumPy to understand the mechanics of how a tree learns, how it measures "impurity", and how it calculates **Feature Importance**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(42)

## 1. Decision Tree Regressor (Minimizing Variance)

For regression tasks, a decision tree splits nodes such that the resulting child nodes have the lowest possible **Mean Squared Error (MSE)** from their local mean.

Mathematically, the algorithm iterates over all features and thresholds to find the split that minimizes the weighted variance of the Left ($L$) and Right ($R$) child nodes:

$$ J(feature, threshold) = \frac{N_L}{N} \text{Var}(Y_L) + \frac{N_R}{N} \text{Var}(Y_R) $$

### Feature Importance
We also calculate feature importance during training. Every time a feature is used to split a node, we calculate the total decrease in impurity (variance) that split brought, weighted by the number of samples in the node. We accumulate this over the whole tree and normalize it.

In [ ]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeRegressorFromScratch:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.feature_importances_ = np.zeros(self.n_features_)
        self.root = self._grow_tree(X, y, 0)
        
        # Normalize feature importances
        if np.sum(self.feature_importances_) > 0:
            self.feature_importances_ /= np.sum(self.feature_importances_)
        
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1:
            return Node(value=np.mean(y))
        
        # Find the best split
        best_feat, best_thresh, best_mse = self._best_split(X, y)
        
        if best_feat is None:
            return Node(value=np.mean(y))
            
        # Feature Importance: Decrease in variance weighted by sample size
        parent_variance = np.var(y)
        impurity_decrease = (parent_variance - best_mse) * n_samples
        self.feature_importances_[best_feat] += impurity_decrease
            
        # Create children
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]
        
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)
        
    def _best_split(self, X, y):
        best_mse = float('inf')
        best_feat, best_thresh = None, None
        n_features = X.shape[1]
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thresh in thresholds:
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]
                
                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue
                    
                mse = self._calculate_split_mse(y, left_idxs, right_idxs)
                
                if mse < best_mse:
                    best_mse = mse
                    best_feat = feat_idx
                    best_thresh = thresh
                    
        return best_feat, best_thresh, best_mse
        
    def _calculate_split_mse(self, y, left_idxs, right_idxs):
        y_l, y_r = y[left_idxs], y[right_idxs]
        var_l, var_r = np.var(y_l) if len(y_l) > 0 else 0, np.var(y_r) if len(y_r) > 0 else 0
        return (len(y_l) / len(y)) * var_l + (len(y_r) / len(y)) * var_r
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

### Train and Visualize Regressor

In [ ]:
X_reg = np.sort(5 * np.random.rand(100, 1), axis=0)
y_reg = np.sin(X_reg).ravel()
y_reg[::5] += 1 * (0.5 - np.random.rand(20))

tree_reg = DecisionTreeRegressorFromScratch(max_depth=3)
tree_reg.fit(X_reg, y_reg)

X_test_reg = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_pred_reg = tree_reg.predict(X_test_reg)

plt.figure(figsize=(8, 5))
plt.scatter(X_reg, y_reg, s=20, edgecolor="black", c="darkorange", label="data")
plt.plot(X_test_reg, y_pred_reg, color="cornflowerblue", label="max_depth=3", linewidth=2)
plt.title("Decision Tree Regression (From Scratch)")
plt.legend()
plt.show()

## 2. Decision Tree Classifier (Minimizing Gini Impurity)

For classification, we minimize **Gini Impurity**. Gini impurity measures how often a randomly chosen element from the set would be incorrectly labeled if it was randomly labeled according to the distribution of labels in the subset.

$$ \text{Gini} = 1 - \sum_{i=1}^{C} (p_i)^2 $$
Where $p_i$ is the probability of an object being classified to a particular class.

In [ ]:
class DecisionTreeClassifierFromScratch:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.feature_importances_ = np.zeros(self.n_features_)
        self.root = self._grow_tree(X, y, 0)
        
        if np.sum(self.feature_importances_) > 0:
            self.feature_importances_ /= np.sum(self.feature_importances_)
        
    def _grow_tree(self, X, y, depth):
        n_samples = X.shape[0]
        n_labels = len(np.unique(y))
        
        if depth >= self.max_depth or n_samples < self.min_samples_split or n_labels == 1:
            return Node(value=self._most_common_label(y))
        
        best_feat, best_thresh, best_gini = self._best_split(X, y)
        
        if best_feat is None:
            return Node(value=self._most_common_label(y))
            
        parent_gini = self._gini(y)
        impurity_decrease = (parent_gini - best_gini) * n_samples
        self.feature_importances_[best_feat] += impurity_decrease
            
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]
        
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)
        
    def _best_split(self, X, y):
        best_gini = float('inf')
        best_feat, best_thresh = None, None
        n_features = X.shape[1]
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thresh in thresholds:
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]
                
                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue
                    
                gini = self._calculate_gini_split(y, left_idxs, right_idxs)
                
                if gini < best_gini:
                    best_gini = gini
                    best_feat = feat_idx
                    best_thresh = thresh
                    
        return best_feat, best_thresh, best_gini
        
    def _gini(self, y_subset):
        _, counts = np.unique(y_subset, return_counts=True)
        probabilities = counts / len(y_subset)
        return 1 - np.sum(probabilities ** 2)
        
    def _calculate_gini_split(self, y, left_idxs, right_idxs):
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        gini_l = self._gini(y[left_idxs])
        gini_r = self._gini(y[right_idxs])
        return (n_l / n) * gini_l + (n_r / n) * gini_r
        
    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

### Train, Visualize, and Inspect Feature Importance

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
# Use all features for training to test feature importance
X_iris = iris.data
y_iris = iris.target

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

clf = DecisionTreeClassifierFromScratch(max_depth=4)
clf.fit(X_train, y_train)
print(f"Accuracy on test set: {np.mean(clf.predict(X_test) == y_test) * 100:.2f}%")

# Plot Feature Importances
plt.figure(figsize=(8, 4))
plt.barh(iris.feature_names, clf.feature_importances_, color='teal')
plt.title("Feature Importances (Calculated from Scratch)")
plt.xlabel("Normalized Gini Importance")
plt.show()